# Laboratório 3: Arquitetura Multi-Agente e LLM-as-a-Judge

**Objetivo:** Implementar o Agente Revisor. Ele interceptará a resposta parcialmente alucinada do Agente Gerador e aplicará um laudo forense cruzando a resposta com o texto recuperado do banco.

In [1]:
# ==========================================
# CÉLULA 1: O TRIBUNAL (AGENTE REVISOR DESACOPLADO)
# ==========================================
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

### Definindo os Prompts (As Leis do Sistema)
Aqui instanciamos o LLM e definimos o prompt do Revisor. Preste atenção em como o prompt do Revisor é desenhado para ser implacável com adições de código.

In [2]:
# 1. Configuração de Caminhos
DIRETORIO_ATUAL = os.getcwd()
RAIZ_DO_PROJETO = os.path.dirname(DIRETORIO_ATUAL)
DB_PATH = os.path.join(RAIZ_DO_PROJETO, "data", "vectordb")
LOG_PATH = os.path.join(RAIZ_DO_PROJETO, "data", "log_resposta_gerador.txt")

### Execução do Fluxo Multi-Agente
Vamos rodar a mesma pergunta. O Gerador vai errar (como você já viu), e o Revisor vai entrar em ação logo em seguida.

In [3]:
# 2. Conectando a Memória e o Cérebro
print("🔌 Conectando ao Banco Vetorial...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print("🧠 Acordando o Agente Revisor (Phi-3)...")
llm = OllamaLLM(model="phi3", temperature=0.1)

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

🔌 Conectando ao Banco Vetorial...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3170.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Acordando o Agente Revisor (Phi-3)...


In [4]:
# 3. Prompt do Juiz
prompt_revisor = PromptTemplate.from_template("""Você é um Auditor Sênior de Qualidade de Software.
Compare a 'Resposta Gerada' com o 'Manual Técnico'. 
A REGRA: Se a 'Resposta Gerada' trouxer QUALQUER código, exemplo ou afirmação que NÃO está escrita palavra por palavra no 'Manual Técnico', você DEVE REPROVAR por vazamento de conhecimento externo.

Se a resposta estiver 100% contida no manual ou disser apenas que não encontrou a informação, responda apenas: 
STATUS: APROVADO.

Se houver adição de conhecimento, preencha estritamente:
STATUS: REPROVADO
1. ONDE ALUCINOU: [Cite o código/texto inventado]
2. POR QUE É UM ERRO: [Explique o desvio da regra do contexto]

Manual Técnico (A VERDADE):
{context}

Pergunta Original: {question}

Resposta Gerada (O RÉU A SER JULGADO):
{resposta_gerada}

Laudo:""")

In [6]:
# 4. Execução Desacoplada
pergunta = "Como faço para dar um duplo clique na tela usando o PyAutoGUI?"
print(f"\n👤 Auditando o fluxo para a pergunta: '{pergunta}'")

# Lendo a Lei (O Contexto do PDF)
documentos = retriever.invoke(pergunta)
contexto_recuperado = formatar_docs(documentos)

# Lendo a Prova do Crime (O que o Gerador fez e salvou no disco)
if not os.path.exists(LOG_PATH):
    raise FileNotFoundError("❌ ERRO: Arquivo de log não encontrado. Rode o Agente Gerador primeiro!")

with open(LOG_PATH, "r", encoding="utf-8") as f:
    resposta_sob_auditoria = f.read()

print("📥 Resposta gerada recuperada com sucesso do disco.")
print("🕵️ Analisando Fidelidade (Faithfulness)...")

# O Julgamento
veredito = llm.invoke(prompt_revisor.format(
    context=contexto_recuperado, 
    question=pergunta, 
    resposta_gerada=resposta_sob_auditoria
))

print("="*60)
print(f"📋 [LAUDO FINAL DO AUDITOR]:\n{veredito}")
print("="*60)


👤 Auditando o fluxo para a pergunta: 'Como faço para dar um duplo clique na tela usando o PyAutoGUI?'
📥 Resposta gerada recuperada com sucesso do disco.
🕵️ Analisando Fidelidade (Faithfulness)...
📋 [LAUDO FINAL DO AUDITOR]:
STATUS: APROVADO

A resposta gerada está contida totalmente dentro dos conteúdos oferecidos pelo manual técnico e não inclui nenhuma informação que não seja explicitamente citada no documento. Portanto, a regra de vazamento de conhecimento externo foi cumprida corretamente.
